# HMM Market-Regime Proof of Concept

Three-state Gaussian hidden Markov model for daily Vietnamese-market OHLCV data. Set `SYMBOL` to any watchlist instrument with complete OHLCV history; states are aligned by Parkinson-volatility emissions before sizing the strategy.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from backend.app.services.data_loader import load_stocks

## 1. Ingest data

In [ ]:
# Load any Vietnamese-market instrument available in the project's OHLCV watchlist.
SYMBOL = "VNINDEX"  # Change to any stock or index symbol in the watchlist.
REQUIRED_FIELDS = ["open", "high", "low", "close", "volume"]

market_data = load_stocks(refresh=False)
available_symbols = market_data.columns.get_level_values("symbol")
if SYMBOL not in available_symbols:
    raise ValueError(f"{SYMBOL!r} is not available in the local market-data watchlist.")

df = market_data.xs(SYMBOL, axis=1, level="symbol").copy()
missing_fields = set(REQUIRED_FIELDS).difference(df.columns)
if missing_fields:
    raise ValueError(f"{SYMBOL!r} is missing OHLCV fields: {sorted(missing_fields)}")

df = df[REQUIRED_FIELDS].dropna().copy()
df.index = pd.to_datetime(df.index)
df = df[~df.index.duplicated(keep="last")].sort_index()

HDF5 not found — fetching from DeltaLake …
Loaded 1000 symbols × 2539 bars from DeltaLake
Saved to stocks_data_latest.h5
Shape: (2539, 1000)  |  2016-09-14 → 2026-09-11


## 2. Feature engineering

In [3]:
# Log returns
df["log_return"] = np.log(df["close"] / df["close"].shift(1))

# Parkinson volatility: intraday range proxy.
df["parkinson_vol"] = np.sqrt(
    (1 / (4 * np.log(2))) * (np.log(df["high"] / df["low"])) ** 2
)

# Rolling volume z-score
df["volume_z"] = (
    (df["volume"] - df["volume"].rolling(20).mean())
    / df["volume"].rolling(20).std()
)

df = df.dropna()
features = ["log_return", "parkinson_vol", "volume_z"]
X = df[features].values

## 3–5. Train, align, and score regimes

In [4]:
# Train three hidden regimes.
model = GaussianHMM(
    n_components=3,
    covariance_type="full",
    n_iter=500,
    random_state=42,
)
model.fit(X)

# HMM state labels are arbitrary. Align them by mean Parkinson volatility.
vol_idx = features.index("parkinson_vol")
state_vols = [model.means_[i][vol_idx] for i in range(3)]
sorted_states = np.argsort(state_vols)

state_map = {
    sorted_states[0]: 0,  # Compression
    sorted_states[1]: 1,  # Expansion
    sorted_states[2]: 2,  # Cascade / tail risk
}

raw_states = model.predict(X)
df["regime"] = [state_map[state] for state in raw_states]

# Real-time continuous probabilities in the aligned state order.
raw_probs = model.predict_proba(X)
aligned_probs = np.zeros_like(raw_probs)
for old_idx, new_idx in state_map.items():
    aligned_probs[:, new_idx] = raw_probs[:, old_idx]

df["p_compression"] = aligned_probs[:, 0]
df["p_expansion"] = aligned_probs[:, 1]
df["p_cascade"] = aligned_probs[:, 2]

Model is not converging.  Current: 15084.847881068166 is not greater than 15085.16532852644. Delta is -0.3174474582738185


## 6–7. Dynamic sizing and performance

In [5]:
# Move to cash when cascade probability spikes; otherwise favor expansion.
df["position_size"] = np.where(
    df["p_cascade"] > 0.50,
    0.0,
    df["p_expansion"] * 1.0 + df["p_compression"] * 0.4,
)

# Tight stops in compression; wider stops in expansion and cascade states.
df["dynamic_stop_mult"] = (
    df["p_compression"] * 1.5
    + df["p_expansion"] * 3.0
    + df["p_cascade"] * 5.0
)

df["strat_returns"] = df["position_size"].shift(1) * df["log_return"]
df["cum_benchmark"] = np.exp(df["log_return"].cumsum())
df["cum_strategy"] = np.exp(df["strat_returns"].cumsum())

print(f"{SYMBOL} Return: {df['cum_benchmark'].iloc[-1] - 1:.2%}")
print(f"Regime Strategy Return: {df['cum_strategy'].iloc[-1] - 1:.2%}")

VNINDEX Return: 166.28%
Regime Strategy Return: 136.28%
